In [43]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import config
import pandas as pd
from pathlib import Path
from src.utils import *
import numpy as np
from functools import reduce
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
import optuna

In [44]:
train = pd.read_csv('data/raw/train.csv')
test = pd.read_csv('data/raw/test.csv')
path = Path('artifacts/oof')

y = train.Irrigation_Need.map(config.TARGET_MAP)

In [45]:
# Level-1 models
select_models = [
    'xgbmV4_oof_proba',
    'realMLP_seed42_v6_oof_proba',
    'Logistic_seed42_v21_oof_proba',
    'Lgbm_seed42_v1_oof_proba',
    'histgbm_seed42_v4_oof_proba',
    'Catgbm_seed42_v10_oof_proba'
]

In [46]:
dfs = []
for name in select_models:
    df = read_csv_file(f'artifacts/oof/{name}.csv')
    
    # drop id after merge key
    df = df.rename(columns=lambda x: f"{name}_{x}" if x != "id" else x)
    dfs.append(df)

# merge all on id
merged = reduce(lambda l, r: pd.merge(l, r, on="id"), dfs)

X_meta = merged.drop(columns=['id'])

Reading file from: artifacts/oof/xgbmV4_oof_proba.csv
Reading file from: artifacts/oof/realMLP_seed42_v6_oof_proba.csv
Reading file from: artifacts/oof/Logistic_seed42_v21_oof_proba.csv
Reading file from: artifacts/oof/Lgbm_seed42_v1_oof_proba.csv
Reading file from: artifacts/oof/histgbm_seed42_v4_oof_proba.csv
Reading file from: artifacts/oof/Catgbm_seed42_v10_oof_proba.csv


In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-4, 100.0, log=True),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "solver": trial.suggest_categorical("solver", ["auto", "svd", "cholesky", "lsqr", "sag"]),
        "max_iter": trial.suggest_int("max_iter", 500, 5000),
        "tol": trial.suggest_float("tol", 1e-5, 1e-2, log=True),
    }

    oof_pred = np.zeros(len(X_meta), dtype=int)
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_meta, y)):
        X_tr = X_meta.iloc[tr_idx]
        X_val = X_meta.iloc[val_idx]
        y_tr = y[tr_idx]
        y_val = y[val_idx]

        model = Ridge(**params)
        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)
        preds = np.clip(np.rint(preds), 0, 2).astype(int)

        oof_pred[val_idx] = preds

        # ---- fold score ----
        fold_score = balanced_accuracy_score(y_val, preds)
        fold_scores.append(fold_score)

        # ---- report to Optuna ----
        trial.report(fold_score, step=fold)

        # ---- pruning condition ----
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # final score
    return balanced_accuracy_score(y, oof_pred)


# -----------------------------
# STUDY WITH MEDIAN PRUNER
# -----------------------------
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    )
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best CV score:", study.best_value)

[I 2026-04-30 23:45:11,536] A new study created in memory with name: no-name-ca035488-9829-4ff4-9513-02d7ff863cf6


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-30 23:45:23,117] Trial 0 finished with value: 0.9782305909414212 and parameters: {'alpha': 0.017670169402947963, 'fit_intercept': True, 'solver': 'sag', 'max_iter': 3205, 'tol': 0.001331121608073689}. Best is trial 0 with value: 0.9782305909414212.


In [ ]:
best_params = study.best_params

oof_pred = np.zeros(len(X_meta), dtype=int)
oof_raw = np.zeros(len(X_meta), dtype=float)

for tr_idx, val_idx in skf.split(X_meta, y):
    X_tr = X_meta.iloc[tr_idx]
    X_val = X_meta.iloc[val_idx]
    y_tr = y[tr_idx]

    model = Ridge(**best_params)
    model.fit(X_tr, y_tr)

    raw_preds = model.predict(X_val)
    oof_raw[val_idx] = raw_preds

    preds = np.clip(np.rint(raw_preds), 0, 2).astype(int)
    oof_pred[val_idx] = preds

oof_score = balanced_accuracy_score(y, oof_pred)
print(f"Final Ridge OOF BACC: {oof_score:.6f}")

# -----------------------------
# FINAL MODEL ON FULL DATA
# -----------------------------
final_model = Ridge(**best_params)
final_model.fit(X_meta, y)

# If you want test predictions later:
# test_raw = final_model.predict(X_test_meta)
# test_pred = np.clip(np.rint(test_raw), 0, 2).astype(int)

In [19]:
oof_pred_raw = meta_model.predict(X_meta)

# convert regression output → class
oof_pred = np.clip(np.round(oof_pred_raw), 0, 2).astype(int)

score = balanced_accuracy_score(y, oof_pred)

print(f"OOF Ridge Stack BACC: {score:.6f}")

OOF Ridge Stack BACC: 0.977566
